# Efficient Estimation of Word Representations in Vector Space (word2vec)

Replication of Mikolov, Chen, Corrado and Dean (2013), *Efficient Estimation of Word
Representations in Vector Space*, ICLR Workshop.

We implement the Skip-gram model with negative sampling and train it on a small public
domain text corpus. The paper's central result is that this shallow model learns
distributed word vectors whose geometry captures semantic and syntactic similarity; we
reproduce this by inspecting nearest neighbors in the learned embedding space.

In [1]:
import re, urllib.request, random, math, collections
import torch, torch.nn as nn
torch.manual_seed(0); random.seed(0)

In [2]:
# Public domain corpus (Project Gutenberg): several full novels for enough text that
# skip-gram learns meaningful word vectors (a few hundred thousand tokens).
urls = [
    "https://www.gutenberg.org/files/1342/1342-0.txt",   # Pride and Prejudice
    "https://www.gutenberg.org/files/1661/1661-0.txt",   # Adventures of Sherlock Holmes
    "https://www.gutenberg.org/files/98/98-0.txt",       # A Tale of Two Cities
    "https://www.gutenberg.org/files/2701/2701-0.txt",   # Moby Dick
    "https://www.gutenberg.org/files/345/345-0.txt",     # Dracula
    "https://www.gutenberg.org/files/1400/1400-0.txt",   # Great Expectations
]
text = ""
for u in urls:
    text += " " + urllib.request.urlopen(u, timeout=30).read().decode("utf-8", "ignore")
words = re.findall(r"[a-z]+", text.lower())
print("total tokens:", len(words))

total tokens: 947545


In [3]:
# Build vocabulary, drop rare words.
MIN_COUNT = 10
counts = collections.Counter(words)
vocab = [w for w, c in counts.items() if c >= MIN_COUNT]
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
raw_data = [stoi[w] for w in words if w in stoi]
V = len(vocab)
# Subsample very frequent words (the word2vec trick) so stopwords do not dominate training.
random.seed(0); total = len(raw_data); idcount = collections.Counter(raw_data); THR = 1e-3
data = [w for w in raw_data if random.random() < math.sqrt(THR * total / idcount[w])]
print("vocab size:", V, "| tokens after subsampling:", len(data), "of", total)

vocab size: 5991 | tokens after subsampling: 558514 of 892089


In [4]:
# Skip-gram pairs (center -> context) within a window, plus a negative-sampling table.
WINDOW = 5
pairs = []
for i, c in enumerate(data):
    for j in range(max(0, i-WINDOW), min(len(data), i+WINDOW+1)):
        if j != i: pairs.append((c, data[j]))
freq = torch.tensor([counts[itos[i]] for i in range(V)], dtype=torch.float)
neg_table = (freq ** 0.75); neg_table /= neg_table.sum()
print("training pairs:", len(pairs))

training pairs: 5585110


In [5]:
DIM, NEG = 100, 5
center_emb = nn.Embedding(V, DIM); context_emb = nn.Embedding(V, DIM)
opt = torch.optim.Adam(list(center_emb.parameters()) + list(context_emb.parameters()), lr=2e-3)

pairs_t = torch.tensor(pairs)
EPOCHS, BS = 10, 2048
for ep in range(EPOCHS):
    perm = torch.randperm(len(pairs_t)); total = 0.0; nb = 0
    for k in range(0, len(pairs_t), BS):
        idx = perm[k:k+BS]
        ctr, ctx = pairs_t[idx, 0], pairs_t[idx, 1]
        neg = torch.multinomial(neg_table, len(idx)*NEG, replacement=True).view(len(idx), NEG)
        v = center_emb(ctr)                                   # (B, D)
        pos_score = (v * context_emb(ctx)).sum(1)             # positive context
        neg_score = torch.bmm(context_emb(neg), v.unsqueeze(2)).squeeze(2)  # negatives
        loss = -(nn.functional.logsigmoid(pos_score).mean() + nn.functional.logsigmoid(-neg_score).mean())
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item(); nb += 1
    print(f"epoch {ep+1}: loss={total/nb:.4f}")

epoch 1: loss=4.1907


epoch 2: loss=1.9211


epoch 3: loss=1.5353


epoch 4: loss=1.4000


epoch 5: loss=1.3404


epoch 6: loss=1.3099


epoch 7: loss=1.2917


epoch 8: loss=1.2793


epoch 9: loss=1.2701


epoch 10: loss=1.2629


In [6]:
# Nearest neighbors by cosine similarity in the learned space.
E = center_emb.weight.detach()
E = E / E.norm(dim=1, keepdim=True)
def neighbors(word, k=8):
    if word not in stoi: return f"'{word}' not in vocab"
    sims = E @ E[stoi[word]]
    top = sims.argsort(descending=True)[1:k+1]
    return [(itos[i.item()], round(sims[i].item(), 3)) for i in top]

for w in ["man", "woman", "father", "money", "night", "sea"]:
    print(f"{w:8s} ->", neighbors(w))

man      -> [('who', 0.534), ('young', 0.449), ('old', 0.447), ('boy', 0.439), ('a', 0.432), ('life', 0.429), ('fellow', 0.426), ('girl', 0.403)]
woman    -> [('daughter', 0.522), ('young', 0.52), ('mother', 0.515), ('lady', 0.451), ('gentleman', 0.445), ('whom', 0.439), ('strong', 0.437), ('handsome', 0.436)]
father   -> [('daughter', 0.576), ('mother', 0.57), ('dear', 0.491), ('her', 0.485), ('name', 0.468), ('marriage', 0.466), ('she', 0.454), ('wife', 0.453)]
money    -> [('witnesses', 0.544), ('your', 0.433), ('pay', 0.408), ('bradstreet', 0.405), ('per', 0.396), ('buy', 0.391), ('piece', 0.39), ('wos', 0.384)]
night    -> [('day', 0.595), ('morning', 0.595), ('sleep', 0.55), ('time', 0.525), ('home', 0.515), ('hour', 0.515), ('until', 0.505), ('jonathan', 0.502)]
sea      -> [('vast', 0.636), ('ship', 0.593), ('whale', 0.572), ('waters', 0.552), ('water', 0.545), ('boat', 0.529), ('seas', 0.528), ('masts', 0.516)]
